In [6]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../7.Near Miss 2017.xlsx"
sheet="DefectDBList (5)"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("Original Shape:", df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan",
    "Not Mentioned",
    "not mentioned",
    "NOT MENTIONED"
]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# RENAME # TO SI NO
# ==========================

for col in df.columns:

    if col.strip()=="#":

        df=df.rename(
            columns={
                col:"SI_No"
            }
        )

        break

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):
        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMN
# ==========================

desc_col=None

for col in df.columns:

    c=col.lower()

    if "description" in c:

        desc_col=col
        break

# ==========================
# REMOVE EMPTY DESCRIPTION
# ==========================

if desc_col:

    before=len(df)

    df=df[
        df[desc_col]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "\nRows Removed (Empty Description):",
        before-len(df)
    )

# ==========================
# FORMAT DATE COLUMNS
# ==========================

date_cols=[]

for col in df.columns:

    c=col.lower()

    if (
        "report_date" in c
        or
        "date_of_occurrence" in c
        or
        c=="report_date"
        or
        c=="date_of_occurrence"
    ):
        date_cols.append(col)

def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned",
        "nan"
    ]:
        return np.nan

    # already formatted → keep as is
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    try:

        d=pd.to_datetime(
            v,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan

for col in date_cols:

    df[col]=df[col].apply(
        clean_date
    )

print("\nFormatted Date Columns:")
print(date_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:", duplicates)

# ==========================
# RESET SI NO
# ==========================

if "SI_No" in df.columns:

    df=df.reset_index(
        drop=True
    )

    df["SI_No"]=range(
        1,
        len(df)+1
    )

print("\nSI No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_7_Near_Miss.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:", output)

Original Shape: (608, 11)

Removed Empty Columns:
[]

Rows Removed (Empty Description): 0

Formatted Date Columns:
['Date_of_occurrence', 'Report_Date']

Duplicates Removed: 0

SI No reordered

Missing Values Summary:
Potential_of_Near_Miss    3
dtype: int64

Final Shape:
(608, 11)

Columns After Cleaning:
['SI_No', 'Date_of_occurrence', 'Reference_No.', 'Report_Date', 'Vessel_Type', 'Office/Vessel', 'Status', 'Report_Type', 'Related_Department', 'Description', 'Potential_of_Near_Miss']

Saved: cleaned_7_Near_Miss.xlsx
